In [ ]:
from option_finder import *
from option_data_plotter import *
try:
    logger
except NameError:
    logger = get_rotating_logger("jupyter", f'logs/put_options.log')

chain_dir = 'chain'
quotes_dir = 'quotes'
data_dir = 'data'
cookie_file = 'cookie.txt'
self = OptionFinder(logger, chain_dir=chain_dir, report_dir=data_dir)

In [ ]:
age_dict = dict([(os.path.basename(f), time.time() - os.path.getmtime(f)) for f in glob(os.path.join(self.chain_dir, '*'))])
symlist = sorted([k for k, v in age_dict.items() if v <= 60], key=age_dict.get)
if len(symlist) > 0:
    print(f'{len(symlist)} symbols, age: {age_dict[symlist[0]]:.01f}" {age_dict[symlist[-1]]:.01f}", {" ".join(symlist)}')
else:
    print('No update in the past 15 minutes.')

### Read downloaded data, check the data age chart to make sure the data are fresh

In [ ]:
df_quotes, df_shortint, df_vola = self.get_quote_df(symlist)
_t0 = time.time()
_df = self.build_option_df(symlist)
_t1 = time.time()
print(f'build_option_df {_t1 - _t0:.1f} seconds')
dfcp = self.concat_put_call_options(_df)
dfcp = bucketize_dte(add_moneyness_columns(dfcp))
_t2 = time.time()
print(f'concat_put_call_options {_t2 - _t1:.1f} seconds')
px.bar(check_data_age(_df), y=['load_age', 'quote_age'], barmode='group', title=f"Data Ages", width=60*len(symlist), height=300).show()
print(f'px.bar {time.time() - _t2:.1f} seconds')
dfcp.loc[:, ['dte', 'expDt']].groupby('dte').first().head(24).tail(20).T

In [ ]:
_g = dfcp[(dfcp.type=='P') & (dfcp.Bid >= 0.5) & (dfcp.OpenInterest >= 100)].groupby('symbol')
_df = pd.DataFrame({'mean spread': _g.pctSpread.mean(), 'median': _g.pctSpread.median()})
px.bar(_df.sort_values(by='mean spread'), barmode='group', width=60*len(symlist))

In [ ]:
try:
    d2e = count_days_from_earning_reports(df_quotes)['earningDays'].to_dict()
except KeyError:
    d2e = {}
print('Days to E:', d2e)

## Put Options
Ignore no-bid or low open interest (minimum open interests is 100)

In [ ]:
dfp = compute_all_time_decay_metrics_for_symbols(dfcp, symlist, 'P', d2e, ignore_no_bid=True, exclude_0dte=True, oi_lb=100)
print('hdte_resid check:', dfp[(dfp.hdte_resid < dfp.resid) & (np.abs(dfp.hdte_resid - dfp.resid) >= 1e-6)].shape)

### Put options with no earning date on or before expiration date

In [ ]:
hdte_resid_ub = 0.75
spread_ub = 5
moneyness_ub = 0.999
premium_lb = 1
delta_lb = -0.15
_filter = (dfp.moneyness <= moneyness_ub) & (dfp.pctSpread <= spread_ub) & (dfp.hdte_resid<=hdte_resid_ub) & (dfp.E.isna() |(dfp.E > dfp.dte))
_filter = _filter & (dfp.premium >= premium_lb) & (dfp.Delta >= delta_lb)
_dfp = dfp[_filter].sort_values(by='hdteProfit', ascending=False)
print(_dfp.shape)
_dfp.head(60)

### Put options including stocks near earning dates

In [ ]:
_moneyness_ub = 0.95
_hdte_resid_ub = 0.7
_filter = (dfp.moneyness <= _moneyness_ub) & (dfp.pctSpread <= 5) & (dfp.hdte_resid<=_hdte_resid_ub)
_dfp = dfp[_filter].sort_values(by='pctProfit', ascending=False)
print(_dfp.shape)
_dfp.head(20)

### Put options for specific symbols

In [ ]:
_filter = dfp.symbol.str.contains('TSM') & (dfp.moneyness <= 1) #& (dfp.pctProfit >= 60) #& (dfp.dte < dfp.E)
#_filter = _filter & (dfp.premium >= 1) #& (dfp.hdte_resid<=0.8)
_dfp = dfp[_filter].sort_values(by='pctProfit', ascending=False)
print(_dfp.shape)
_dfp.head(25)

### Put options: top 500 in-the-money

In [ ]:
px.scatter(dfp[dfp.moneyness <= 1].sort_values(by='pctProfit', ascending=False).head(500), x='hdte_resid', y='pctProfit', color='symbol', height=600)

### The End